# 04 - Attendance anomalies without labels (Day 5)
Labels are used **only** to measure, after the flags are saved.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50); pd.set_option('display.max_colwidth', 120)
from cortexhr import config as C, data_loading as D, audit as A, features as F, modeling as M, evaluation as E, reporting as R, runs

In [2]:
from cortexhr import attendance as T
att = D.load_sheet(C.SHEET_ATT); att.describe().T[['min', 'mean', 'max']]

                                            min                        mean                  max
month                       2025-04-01 00:00:00  2025-12-21 12:49:56.157514  2026-09-01 00:00:00
job_level                                   1.0                    2.721079                  7.0
scheduled_days                              1.0                   20.459872                 23.0
present_days                                0.0                   19.103284                 23.0
wfh_days                                    0.0                    8.041423                 23.0
half_days_unpaid                            0.0                    0.089658                  4.0
paid_leave_days                             0.0                    0.972203                 23.0
unpaid_absence_days                         0.0                    0.294999                 22.0
longest_unpaid_streak_days                  0.0                    0.263524                 22.0
late_arrivals                 

## Model: IsolationForest (3% contamination) + two simple rules

In [3]:
flags = T.detect(att, contamination=0.03, seed=C.SEED)
flags[['iforest_flag', 'iforest_raw_only_flag', 'rule_zero_attendance', 'rule_implausible_hours', 'rule_flag', 'flagged']].sum()

iforest_flag              221
iforest_raw_only_flag     221
rule_zero_attendance       50
rule_implausible_hours     44
rule_flag                  94
flagged                   252
dtype: int64

In [4]:
T.assert_no_labels(T.RAW_FEATURES + T.HISTORY_FEATURES); print('features:', T.RAW_FEATURES + T.HISTORY_FEATURES)

features: ['scheduled_days', 'present_days', 'half_days_unpaid', 'paid_leave_days', 'unpaid_absence_days', 'longest_unpaid_streak_days', 'late_arrivals', 'avg_hours_worked', 'overtime_hours', 'attendance_rate', 'rate_vs_own_median', 'late_vs_own_median', 'hours_vs_own_median']


## Ranking flags for a limited review budget

In [5]:
T.review_queue(flags, 15)[['attendance_feature_id', 'employee_id', 'month', 'flag_reason', 'priority_score', 'review_rank']]

     attendance_feature_id employee_id      month                     flag_reason  priority_score  review_rank
2818              AF-02819    EMP-0318 2026-06-01  rule: zero attendance in month        1.769660          1.0
5874              AF-05875    EMP-0582 2026-01-01  rule: zero attendance in month        1.768901          2.0
5917              AF-05918    EMP-0585 2025-09-01  rule: zero attendance in month        1.767260          3.0
6634              AF-06635    EMP-0633 2026-08-01  rule: zero attendance in month        1.766678          4.0
3169              AF-03170    EMP-0356 2025-07-01  rule: zero attendance in month        1.766277          5.0
6442              AF-06443    EMP-0619 2026-08-01  rule: zero attendance in month        1.765920          6.0
2086              AF-02087    EMP-0246 2025-08-01  rule: zero attendance in month        1.765907          7.0
6801              AF-06802    EMP-0646 2026-06-01  rule: zero attendance in month        1.765186          8.0
5

## Measurement against hidden labels
The flags are saved first (`runs.run_attendance`), then labels are read. If the labels file is not in `04_Validation_Vault/` the measurement is reported as NOT MEASURED.

In [6]:
res = runs.run_attendance(write=False)
print('labels file:', res['labels_path'] or 'NOT SUPPLIED -> monthly precision/recall not measured')
res['measured']['overall'].round(3) if res['measured'] else res['benchmark']

labels file: NOT SUPPLIED -> monthly precision/recall not measured


                                                                 task                                                                                                                 features  \
0           Attendance anomalies (Isolation Forest, 3% contamination)  scheduled_days, present_days, half_days_unpaid, paid_leave_days, unpaid_absence_days, longest_unpaid_streak_days, la...   
1  Payroll anomalies (Isolation Forest on ratios, 2.5% contamination)                                             gross_vs_fixed, ded_ratio, net_ratio, overtime_hours_paid, dup_flag, net_pay   

   precision  recall  flagged  injected  
0      0.403   0.593      221       150  
1      0.217   0.400      184       100  

In [7]:
res['measured']['by_type'] if res['measured'] and res['measured']['by_type'] is not None else 'anomaly-type breakdown needs the labels file'

'anomaly-type breakdown needs the labels file'

## Supplementary: labelled DAILY sample (different table, measurement only)

In [8]:
res['daily_proxy'].round(3)

                   detector  flagged  injected  caught  precision  recall
0     Daily IsolationForest       60       120      47      0.783   0.392
1  Daily rule (hours/clock)       93       120      90      0.968   0.750
2            Daily combined      107       120      93      0.869   0.775

Report: `reports/attendance_anomaly_report.md`.